# Task 3: A/B Hypothesis Testing

## Objective
Statistically validate or reject key hypotheses about risk drivers, forming the evidence base for ACIS's new segmentation and pricing strategy.

**KPIs defined:**
- **Claim Frequency**: Proportion of policies with at least one claim.
- **Claim Severity**: Average claim amount, given a claim occurred.
- **Margin**: TotalPremium − TotalClaims.

In [ ]:
import os
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Add src to path
sys.path.append(os.path.abspath('../src'))

from data_loader import load_raw_data, preprocess
from hypothesis_tests import (
    test_province_risk, test_zipcode_risk, 
    test_margin_zipcode, test_gender_risk, 
    build_results_table
)

pd.set_option('display.max_columns', None)
print("Environment ready.")

## 1. Load and Preprocess Data

In [ ]:
DATA_PATH = r'c:\KAIM\MachineLearningRating_v3.txt'

try:
    df_raw = load_raw_data(DATA_PATH)
    df = preprocess(df_raw)
    print(f"Data loaded successfully: {df.shape[0]:,} rows")
except FileNotFoundError:
    print("Data file not found. Please ensure the path is correct.")

## 2. Hypothesis Testing

### H₀: No risk differences across provinces
We test Claim Frequency (Chi-Squared) and Claim Severity (T-test) across the top provinces.

In [ ]:
# To keep it manageable, we test the top 2 provinces by policy count
top_provinces = df['Province'].value_counts().index[:2].tolist()
df_prov = df[df['Province'].isin(top_provinces)]

prov_results = test_province_risk(df_prov)
print(f"Completed {len(prov_results)} province tests.")

### H₀: No risk differences between zip codes
We compare the two most populated zip codes.

In [ ]:
zip_results = test_zipcode_risk(df)
print(f"Completed {len(zip_results)} zip code risk tests.")

### H₀: No significant margin (profit) difference between zip codes

In [ ]:
margin_results = test_margin_zipcode(df)
print(f"Completed {len(margin_results)} margin tests.")

### H₀: No significant risk difference between Women and Men

In [ ]:
gender_results = test_gender_risk(df)
print(f"Completed {len(gender_results)} gender tests.")

## 3. Results Summary

In [ ]:
all_tests = prov_results + zip_results + margin_results + gender_results
summary_df = build_results_table(all_tests)
summary_df

## 4. Business Interpretations

Based on the results table above:

1. **Provinces**: If p < 0.05, we reject H₀. This implies geographic location is a significant driver of risk (either frequency or severity), and premiums should be adjusted by province.
2. **Zip Codes**: Significant differences in zip codes (p < 0.05) suggest that even within provinces, hyper-local risk factors exist (e.g., crime rates, traffic density).
3. **Margin**: If margin differences are significant, it indicates that current pricing is not effectively neutralizing the risk differences between locations, leading to "under-priced" or "over-priced" areas.
4. **Gender**: If gender risk is significant, it justifies gender-based segmentation (where legally permitted) to better reflect the risk profile of different demographic groups.